# RAG，Retrieval-Augmented Generation，检索增强生成

## LLM的幻觉
模型生成的内容与事实不符、逻辑矛盾或无法被输入/知识验证的现象。

核心就一句话：模型输出了一本正经但实际错误的东西。

## 外挂知识库
把LLM不知道的知识作为提示词的一部分发送给LLM

## 知识切分
与用户问题相关的知识其实只是一小部分。可以把知识库切分成一个个的知识片段，当用户提问时，只携带与问题相关的知识片段

## 知识检索
从语义分析来检索，找到语义上与用户问题最接近的知识片段。

### 向量相似度
把用户的问题（一句话）转为向量，把知识片段（一段话）转为向量，然后就可以通过比较向量相似度来判断两者的含义是否接近了。

向量之间的距离常见的计算方法有：
- 余弦相似度（Cosine similarity）：两个向量之间的夹角。
- 欧氏距离（Euclidean distance ）：两点之间的直线距离。
- 点积（Dot product）：一个向量在另一个向量上的投影量

通常，两个向量之间距离越近，我们认为两个向量的相似度越高（距离值越小，相似度越高）

把文本转为向量，就可以通过向量距离来判断文本的相似度了。

### 向量数据库
向量数据库的主要作用有两个：
- 存储向量数据（知识片段）
- 基于相似度检索向量数据（知识片段）

## RAG
RAG，Retrieval-Augmented Generation，关键词：
- 检索：就是利用向量相似度检索知识片段
- 增强：用检索到的知识片段增强模型，减少幻觉
- 生成：模型基于知识片段生成答案

### RAG核心流程
综上所述，RAG分为两大阶段：
- 离线阶段：负责构建知识库
  1. 知识加载：读取各种来源的知识库数据，解析、清洗数据，形成文档（Documents）。
  2. 知识切分：把清洗后的知识文档切分成一个个的知识片段（Chunks）
  3. 向量化：利用向量模型将知识片段转为向量，使得语义相近的文本在该向量空间中彼此邻近
  4. 存储向量：处理好的向量及对应的知识片段存入向量数据库
- 在线阶段：负责检索知识，生成回答
  1. 问题向量化：利用向量模型（必须与离线阶段同一个模型）将用户问题转为向量
  2. 知识召回：在向量数据库中检索与问题向量最相似的文档的TopN
  3. 生成回答：把检索到的知识片段与用户提问组织为新提示词，发给大模型，生成回答

### LangChain的RAG组件
主要的组件包括：
- 离线阶段
  1. 知识加载：LangChain提供了Document Loader组件
  2. 知识切分：LangChain提供了Text Splitter组件
  3. 向量化：LangChain提供了Embeddings接口，兼容各类向量模型
  4. 存储向量：LangChain提供了VectorStore接口，兼容各类向量数据库
- 在线阶段：负责检索知识，生成回答
  1. 问题向量化：同样基于Embeddings模型接口
  2. 知识召回：LangChain提供了Retrievers组件，简化知识片段的检索操作
  3. 生成回答：直接调用模型即可

### RAG代码预览

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# ===============一、构建知识库阶段====================
# =============1.加载文档===============
loader = PyPDFLoader(
    "resources/贵州茅台研报.pdf",
    mode="single", # single \ page
)
docs = loader.load()
print(f"文档数量:{len(docs)}")

# =============2.切分文档===============
# 创建递归切分器
splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=150)
# 切分文档
chunks = splitter.split_documents(docs)
print(f"分块数量：{len(chunks)}")

# =============3.向量化===============
# 向量模型，这里用阿里云的向量模型
embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 创建向量库，需要指定向量模型，存储文档时会自动调用向量模型完成向量化
vectorstore = InMemoryVectorStore(embedding=embeddings)
# 添加文档
vectorstore.add_documents(chunks)

# ==============二、在线问答阶段================
llm = init_chat_model("deepseek-chat")

def try_rag(query: str):
    # 1.检索文档（VectorStore会自动把问题向量化，召回相关知识片段）
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    # 2.拼接上下文提示词
    content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f"""你基于我提供的报告回答用户问题，报告中没提及的就说不知道，不要自己编造答案.
    report: ```{content}```
    query: {query}"""
    # 3.调用模型，生成答案
    response = llm.invoke(prompt)
    return response.content